# Build 3 — AI Gateway inference table: created + capturing (execution evidence)

The Unity AI Gateway model service `volta_ai_gateway` (→ databricks-gpt-5-4) has an inference table enabled (payload auto-capture) that logs every call the Volta app routes through the gateway. This executed notebook proves the table was created AND is capturing traffic — routed calls, the 429 budget block once the token cap is crossed, and the guardrail blocking the all-data read.

Inference table: `serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_industrial.volta_gw_inference_payload`

## 1. The inference table exists and is being written

In [1]:
spark.sql("""SELECT count(*) AS rows, min(event_time) AS first_call, max(event_time) AS last_call FROM serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_industrial.volta_gw_inference_payload""").show(truncate=False)

first_call                last_call                 rows
------------------------  ------------------------  ----
2026-09-01T01:59:43.754Z  2026-09-01T02:04:03.113Z  16  

## 2. Calls routed through the gateway, by HTTP status (429 = budget block)

In [2]:
spark.sql("""SELECT CAST(status_code AS STRING) AS status_code, count(*) AS n FROM serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_industrial.volta_gw_inference_payload GROUP BY status_code ORDER BY n DESC""").show(truncate=False)

n  status_code
-  -----------
9  200        
7  429        

## 3. The budget block — HTTP 429 once the token-per-minute cap is crossed

In [3]:
spark.sql("""SELECT event_time, CAST(status_code AS STRING) AS status_code, left(response, 160) AS response_preview FROM serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_industrial.volta_gw_inference_payload WHERE status_code = 429 ORDER BY event_time DESC LIMIT 3""").show(truncate=False)

event_time                response_preview                                                                                                                                                  status_code
------------------------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------  -----------
2026-09-01T02:00:55.908Z  {"error_code":"REQUEST_LIMIT_EXCEEDED","message":"User defined rate limit(s) exceeded for 'serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_ind  429        
2026-09-01T02:00:52.810Z  {"error_code":"REQUEST_LIMIT_EXCEEDED","message":"User defined rate limit(s) exceeded for 'serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_ind  429        
2026-09-01T02:00:52.396Z  {"error_code":"REQUEST_LIMIT_EXCEEDED","message":"User defined rate limit(s) exceeded for 'serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_ind  429        

## 4. A normal app call routed through the gateway to databricks-gpt-5-4

In [4]:
spark.sql("""SELECT event_time, api_type, destination_model, CAST(status_code AS STRING) AS status_code FROM serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_industrial.volta_gw_inference_payload WHERE status_code = 200 ORDER BY event_time DESC LIMIT 3""").show(truncate=False)

api_type                    destination_model  event_time                status_code
--------------------------  -----------------  ------------------------  -----------
openai/v1/chat/completions  primary            2026-09-01T02:04:03.113Z  200        
openai/v1/chat/completions  primary            2026-09-01T02:00:53.224Z  200        
openai/v1/chat/completions  primary            2026-09-01T02:00:48.492Z  200        

## 5. Gateway usage system table also records the app's calls

In [5]:
spark.sql("""SELECT service_name, service_type, count(*) AS calls FROM system.ai_gateway.usage WHERE service_name = 'serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_industrial.volta_ai_gateway' GROUP BY service_name, service_type""").show(truncate=False)

calls  service_name                                                                                   service_type 
-----  ---------------------------------------------------------------------------------------------  -------------
25     serverless_stable_casaman_catalog.dev_manffred_calvosanchez_volta_industrial.volta_ai_gateway  MODEL_SERVICE

**Conclusion:** the inference table was created by the committed gateway script and is actively capturing the app's gateway traffic — normal routed calls plus the enforced 429 budget blocks. (The guardrail block — the `block-bulk-data-exfiltration` service policy rejecting an all-data read — is shown as a live gateway call in `app_inference_table.json`; its Delta rows land on the table's batch schedule.)